# PANDAS MASTERY — CHAPTERS 64–76 (COURSE FINALE)

```
PANDAS MASTERY
Current Chapters: 64 (Real-World Scenarios), 65 (Pandas vs SQL), 66 (Pandas vs Polars),
                  67 (Pandas vs Spark), 68 (Performance Engineering),
                  69 (Advanced Patterns), 70 (Business Analytics: RFM/Cohort/Retention),
                  71 (ML Feature Engineering Masterclass), 72 (Experimentation),
                  73 (Model Error Analysis), 74 (Model Monitoring), 75 (Capstone),
                  76 (Mock Interview)
Progress:         76 / 76 chapters -- COURSE COMPLETE
Completed:        Ch1-63
Remaining:        --
```

The last thirteen chapters. 64-67 are scenario/judgment chapters (when to reach for Pandas vs
SQL vs Polars vs Spark). 68-74 are hands-on, code-verified: performance profiling, advanced
patterns, a real RFM + cohort + retention analysis, a full ML feature engineering catalog,
experiment tracking, model error analysis, and model monitoring/drift detection. 75 is a real
capstone assignment -- attempt it yourself, it's deliberately left unsolved. 76 opens the mock
interview format for a follow-up conversation. Every runnable number below was executed and
verified before this notebook was written.


In [ ]:
import pandas as pd
import numpy as np
print("pandas version:", pd.__version__)
print("numpy version :", np.__version__)


---
# CHAPTER 64 — Real-World Data Scenarios

Attempt each yourself before reading the model answer.

## Scenario 1: A customer table contains duplicate customer IDs
**What would you investigate?** Whether they're exact duplicates (every column identical --
likely a re-ingestion bug, generally safe to drop) or partial duplicates with differing values
in some columns (could be legitimate updates over time -- you likely want the most recent row,
not a blind drop). Also check whether `customer_id` is supposed to be a unique key at all -- if
so, the duplication itself is worth flagging as an upstream data quality bug (Chapter 13, 37).

## Scenario 2: A merge increases rows by 10x
**Why?** The join key has duplicate values on **both** sides -- a many-to-many relationship
producing `m x n` output rows per duplicated key value, not a bug (Chapter 25). Check
`left[key].duplicated().sum()` and `right[key].duplicated().sum()` before trusting any large
merge; `validate=` (Chapter 24) would have caught this immediately if the relationship was
assumed to be one-to-one or one-to-many.

## Scenario 3: A dataset contains 40% missing values
**What do you do?** Investigate the missingness mechanism (MCAR/MAR/MNAR, Chapter 12) before
choosing a strategy -- 40% is too high to impute blindly without understanding why. Check
whether missingness correlates with other columns or the target; consider an explicit
"was_missing" indicator feature if the pattern seems informative; consider whether the column
is even usable at that missingness rate versus dropping it entirely. Never drop 40% of *rows*
just to keep the column complete -- that discards far more information than the missingness
itself costs.

## Scenario 4: A date column contains five different formats
**How do you clean it?** `pd.to_datetime(col, format="mixed", errors="coerce")` (Chapter 8/37,
and directly demonstrated with real data in this course's own Chapter 37 cleaning project) --
`format="mixed"` infers each element's format individually rather than requiring one consistent
format across the whole column; `errors="coerce"` converts genuinely unparseable values to
`NaT` rather than aborting the whole operation, so they can be investigated and handled
explicitly afterward.

## Scenario 5: A categorical feature contains `India`/`india`/`INDIA`/` India`
**How do you standardize it?** `.str.strip().str.lower()` (or `.str.title()` for a consistent
display case) -- exactly Chapter 16's string-cleaning chain, and exactly the pattern this
course's Chapter 56 churn project applied for real to a `region` column with the identical
fragmentation shape (`'East'`/`'east'`/`'EAST'`/`'east '`).

## Scenario 6: A dataset has 50 million rows
**How do you process it?** First ask whether it genuinely needs to be fully in memory at once --
`chunksize`-based streaming with cross-chunk accumulation (Chapter 44) handles many aggregation
tasks without ever holding the full dataset in RAM. If the required computation is more complex
than simple accumulation supports, or genuinely needs repeated random access across the whole
dataset, evaluate Polars (faster, lower memory, single machine) or DuckDB (SQL-native,
excellent for this exact scale on one machine) before reaching for genuinely distributed tools
like Spark, which add real operational complexity that isn't justified until the data or
compute truly exceeds single-machine capacity (Chapters 66-67).


---
# CHAPTERS 65–67 — Pandas vs SQL, Polars, and Spark

## The decision framework, not just a feature comparison
The real skill being tested here isn't "know the syntax of four tools" -- it's **knowing when to
stop using Pandas**, which is itself a mark of engineering maturity many candidates lack.

| | Pandas | SQL (on a database) | Polars | Spark |
|---|---|---|---|---|
| **Execution model** | Eager, in-process, single-threaded per operation (mostly) | Query planner, often set-based, can leverage indexes | Eager or lazy, multi-threaded by default, Rust-backed | Distributed, lazy, multi-machine |
| **Typical data scale** | Fits comfortably in one machine's RAM (up to low tens of GB) | Whatever the database engine + hardware can index/scan efficiently | Larger-than-Pandas-comfortable, still single-machine | Genuinely exceeds single-machine memory/compute |
| **Where computation happens** | Client-side, in your Python process | Server-side, in the database | Client-side, but far more efficient than Pandas | Across a cluster |
| **API style** | Imperative, method-chaining | Declarative (`SELECT`...`FROM`...`WHERE`) | Similar to Pandas, with a lazy/expression API option | DataFrame API similar to Pandas, or SQL |
| **ML ecosystem integration** | Native, direct handoff to scikit-learn/PyTorch (this entire course's central theme) | Requires export to a DataFrame library first | Growing, via `.to_pandas()` or native integrations | `MLlib`, or export to a DataFrame library |
| **When it's the right choice** | Exploration, feature engineering, moderate-scale analysis, the default choice until proven insufficient | Data already lives in a database, and the operation is naturally expressible as a query (aggregation, filtering) close to the data source | Pandas is becoming too slow/memory-hungry but the data still fits on one machine | Data or compute genuinely exceeds one machine's capacity, or you're already in a Spark-based data platform |

## Concrete "when NOT to use Pandas" signals
- Your dataset doesn't comfortably fit in available RAM even with dtype optimization and
  chunking (Chapters 43-44) -- consider Polars or DuckDB first, since they're often simpler
  operational additions than Spark.
- Your data already lives in a well-indexed SQL database, and the operation you need is a
  straightforward filter/aggregate -- running that query in the database (`WHERE`, `GROUP BY`)
  is almost always faster than pulling everything into Pandas first and filtering there.
- Your organization's data platform is already Spark-based, and your output needs to integrate
  with distributed downstream jobs -- fighting the existing platform to force everything through
  Pandas creates unnecessary friction, even if your specific dataset would technically fit.

## The Pandas to SQL clause mapping (fully covered with working code in Chapter 45)
`SELECT` -> column selection; `WHERE` -> boolean filtering (before aggregation); `GROUP BY` ->
`.groupby()`; `JOIN...ON` -> `.merge()`; `ORDER BY` -> `.sort_values()`; `DISTINCT` ->
`.drop_duplicates()`; `HAVING` -> post-aggregation filtering (`.agg(...).query(...)`); `CASE
WHEN` -> `np.where()`/`np.select()`.

### Interview Insight
> "When would you recommend moving a Pandas pipeline to Polars or Spark?"
Polars first, if the bottleneck is single-machine performance/memory but the data still fits on
one machine -- it's a far simpler operational change (same general DataFrame paradigm, often a
near-drop-in API) than adopting Spark. Spark only once the data or required compute genuinely
exceeds what any single machine can handle, or the existing data platform is already
Spark-centric -- Spark's distributed-systems overhead (cluster management, network shuffles,
different debugging model) isn't worth paying unless the scale genuinely demands it.


---
# CHAPTER 68 — Performance Engineering

## Concept
Formal profiling, not folklore -- measuring exactly where time and memory go, extending Chapter
43's individual measurements into a systematic investigation habit.


In [ ]:
import timeit

n = 200_000
df = pd.DataFrame({
    "id": range(n),
    "value": np.random.default_rng(0).uniform(0, 1000, n),
    "category": np.random.default_rng(1).choice(list("ABCDE"), size=n),
})

def naive_filter_loop():
    result = []
    for v in df["value"]:
        if v > 500:
            result.append(v)
    return result

def vectorized_filter():
    return df[df["value"] > 500]

t_loop = timeit.timeit(naive_filter_loop, number=1)
t_vec = timeit.timeit(vectorized_filter, number=1)
print(f"naive loop filter: {t_loop:.4f}s   vectorized filter: {t_vec:.4f}s   ratio: {t_loop/t_vec:.0f}x")


### Expected Output (verified on pandas 3.0.2)
```
naive loop filter: 0.0252s   vectorized filter: 0.0145s   ratio: 2x
```

### A genuinely useful "not every measurement gives a dramatic result" finding
Notice this ratio (`2x`) is far more modest than Chapter 43's `apply(axis=1)` result
(`1031x`). Iterating a Series with a plain Python `for` loop and appending to a list is slow,
but **not catastrophically** slow the way `DataFrame.apply(axis=1)` is -- `apply(axis=1)`
specifically pays the extra cost of constructing a full `Series` object per row, which a simple
column iteration doesn't. **This is an important, honest calibration**: not every
"vectorize this" opportunity is a 1000x win; profiling tells you which specific inefficiency in
your actual code is worth fixing first, rather than assuming every loop is equally catastrophic.

## Column pruning -- the cost of copying unused columns


In [ ]:
import time

t0 = time.perf_counter()
_ = df[["id", "value"]].copy()
t1 = time.perf_counter()
_ = df.copy()
t2 = time.perf_counter()

print(f"copy 2 columns: {t1-t0:.5f}s")
print(f"copy ALL columns: {t2-t1:.5f}s")


### Expected Output (verified on pandas 3.0.2 -- timings vary by machine, the direction is the point)
```
copy 2 columns: 0.00200s
copy ALL columns: 0.01542s
```

### Why column selection early in a pipeline is a real, measurable optimization
Copying only the 2 needed columns was roughly 7-8x faster than copying all 3 (including the
`category` column, whose Arrow-backed string storage carries real per-copy overhead -- Chapter
17's category-vs-str memory lesson, now showing up as a copy-time cost too). In a real pipeline
processing many DataFrames with many unused columns, selecting down to only the columns a given
step actually needs -- before expensive operations like `.copy()`, `.merge()`, or `.groupby()`
-- compounds into a real difference at scale, not just a memory optimization.

### Interview Insight
> "You're optimizing a slow Pandas pipeline. Besides vectorizing loops, what's another
> low-effort, high-value change to look for?"
Selecting only the columns actually needed for each pipeline stage, as early as possible --
carrying unused columns through merges, groupbys, and copies wastes both memory and time
proportional to those columns' size, and this is often a much smaller code change than
rewriting algorithmic logic, verified above as a real, non-trivial speedup for something as
simple as a `.copy()`.


---
# CHAPTER 69 — Advanced Pandas Patterns

## Concept
Recurring, real-world patterns worth having ready-made: latest/first-record selection,
business-rule deduplication, and conditional aggregation.


In [ ]:
events = pd.DataFrame({
    "user_id": [1, 1, 1, 2, 2, 3],
    "event_time": pd.to_datetime(["2023-01-01", "2023-01-05", "2023-01-10", "2023-02-01", "2023-02-10", "2023-03-01"]),
    "status": ["pending", "shipped", "delivered", "pending", "shipped", "delivered"],
})

# --- Latest record per entity ---
latest = events.sort_values("event_time").groupby("user_id").tail(1)
print("Latest record per user:\n", latest)

# --- First observation per entity ---
first = events.sort_values("event_time").groupby("user_id").head(1)
print("\nFirst record per user:\n", first)


### Expected Output (verified on pandas 3.0.2)
```
Latest record per user:
    user_id event_time     status
2        1 2023-01-10  delivered
4        2 2023-02-10    shipped
5        3 2023-03-01  delivered

First record per user:
    user_id event_time     status
0        1 2023-01-01    pending
3        2 2023-02-01    pending
5        3 2023-03-01  delivered
```

### `sort_values().groupby().tail(1)`/`.head(1)` -- the standard, efficient latest/first pattern
Sort once globally, then take the head/tail within each group -- this reuses exactly Chapter
63's `groupby().head(n)` top-N idiom, specialized to `n=1`. This is the standard way to answer
"what's each entity's most recent status" (a customer's current order status, a device's latest
sensor reading) without a slower `.apply()`-based per-group max lookup.

## Deduplication by a genuine business rule, not just "keep first"


In [ ]:
priority = {"delivered": 3, "shipped": 2, "pending": 1}
events["priority"] = events["status"].map(priority)

dedup = (
    events
    .sort_values(["user_id", "event_time", "priority"], ascending=[True, False, False])
    .groupby("user_id")
    .head(1)
)
print("Business-rule dedup (most recent, highest priority on tie):\n", dedup)


### Expected Output (verified on pandas 3.0.2)
```
Business-rule dedup (most recent, highest priority on tie):
    user_id event_time     status  priority
2        1 2023-01-10  delivered         3
4        2 2023-02-10    shipped         2
5        3 2023-03-01  delivered         3
```

### Why explicit priority mapping beats implicit alphabetical/positional tie-breaking
Sorting by `event_time` descending alone could leave ties broken by whatever order rows happen
to appear in -- mapping `status` to an explicit numeric `priority` and including it as a
secondary sort key makes the tie-breaking rule **visible and intentional** in the code, rather
than an accidental artifact of row order. This is the professional-quality version of "keep
the most recent, and if there's a tie, prefer the more advanced status" -- a real business rule,
not just `drop_duplicates(keep='first')`'s arbitrary default.

## Conditional aggregation -- computing two different sums from the same group, based on a condition


In [ ]:
sales = pd.DataFrame({
    "region": ["E", "E", "W", "W"],
    "amount": [100, 200, 150, 50],
    "is_return": [False, True, False, True],
})

conditional_agg = sales.groupby("region").apply(
    lambda g: pd.Series({
        "gross_sales": g.loc[~g["is_return"], "amount"].sum(),
        "returns": g.loc[g["is_return"], "amount"].sum(),
    }),
    include_groups=False,
)
print(conditional_agg)


### Expected Output (verified on pandas 3.0.2)
```
        gross_sales  returns
region
E               100      200
W               150       50
```

### Reading this as a real business insight, not just syntax
Region E's `returns` (`200`) actually **exceed** its `gross_sales` (`100`) -- a genuine red flag
(more returned value than sold value within this data) that a single blended `amount.sum()`
would completely hide (it would show a misleadingly modest net of `100 - 200 = -100`, without
making clear *why* it's negative). Conditional aggregation -- computing gross sales and returns
as two **separate** numbers, not one net figure -- surfaces exactly this kind of
otherwise-invisible pattern.

### Interview Insight
> "How would you compute both 'gross sales' and 'returns' as separate columns within the same
> groupby, when they're both derived from the same `amount` column under different conditions?"
Use `.groupby(...).apply(lambda g: pd.Series({...}))`, computing each output value via a
boolean-masked sum (`g.loc[condition, 'amount'].sum()`) inside the lambda -- this returns a
named `Series` per group that becomes multiple output columns, letting you compute several
conditionally-different aggregates from the same underlying column in one grouped pass.


---
# CHAPTER 70 — Business Analytics with Pandas: RFM, Cohort, and Retention Analysis

## RFM Analysis — Recency, Frequency, Monetary value
The classic customer-segmentation framework: how recently did they buy, how often, how much --
three numbers that together characterize customer value far better than any one alone.


In [ ]:
orders = pd.DataFrame({
    "customer_id": [1, 1, 2, 2, 2, 3, 4, 4, 5],
    "order_date": pd.to_datetime([
        "2023-01-05", "2023-03-10", "2023-01-15", "2023-02-20", "2023-03-25",
        "2023-01-01", "2023-02-01", "2023-02-15", "2023-03-01",
    ]),
    "amount": [50, 80, 30, 45, 60, 200, 20, 25, 90],
})
reference_date = pd.Timestamp("2023-04-01")

rfm = orders.groupby("customer_id").agg(
    recency=("order_date", lambda s: (reference_date - s.max()).days),
    frequency=("order_date", "count"),
    monetary=("amount", "sum"),
)
print("RFM table:\n", rfm)


### Expected Output (verified on pandas 3.0.2)
```
RFM table:
              recency  frequency  monetary
customer_id
1                 22          2       130
2                  7          3       135
3                 90          1       200
4                 45          2        45
5                 31          1        90
```

### Converting raw RFM into 1-3 scores via `pd.qcut`


In [ ]:
rfm["r_score"] = pd.qcut(rfm["recency"], 3, labels=[3, 2, 1])          # LOWER recency (more recent) -> HIGHER score
rfm["f_score"] = pd.qcut(rfm["frequency"].rank(method="first"), 3, labels=[1, 2, 3])
rfm["m_score"] = pd.qcut(rfm["monetary"], 3, labels=[1, 2, 3])
print(rfm)


### Expected Output (verified on pandas 3.0.2)
```
             recency  frequency  monetary r_score f_score m_score
customer_id
1                 22          2       130       3       2       2
2                  7          3       135       3       3       3
3                 90          1       200       1       1       3
4                 45          2        45       1       3       1
5                 31          1        90       2       1       1
```

### Reading customer 3 as a real business insight `pd.qcut` surfaces
Customer 3 has the **highest** monetary value in the dataset (`200`, `m_score=3`) but the
**worst** recency (`90` days, `r_score=1`) and lowest frequency (`f_score=1`) -- a classic **"was
valuable, now at risk"** profile: a high-spending customer who hasn't purchased in a long time.
This exact combination is what real RFM-based retention campaigns target specifically (win-back
offers for high-value-but-lapsed customers), and it's only visible because RFM scores each
dimension **independently** rather than blending everything into one composite number that
would hide this tension between "valuable" and "at risk."

### `rank(method='first')` before `qcut` on `frequency` — why it's needed here
`pd.qcut` requires enough distinct values to form the requested number of bins; `frequency` has
only 3 distinct values (`1`, `2`, `3`) across 5 customers, which can cause `qcut` to fail or
produce non-bins with ties clustered at bin edges. `.rank(method='first')` breaks ties by
original row order first, producing enough distinct rank values for `qcut` to bin cleanly --
a real, practical workaround worth knowing rather than being surprised by when `qcut` complains
about a low-cardinality column.

## Cohort Analysis and Retention — grouping customers by signup period, tracking activity over time


In [ ]:
customers = pd.DataFrame({
    "customer_id": [1, 2, 3, 4, 5],
    "signup_month": pd.to_datetime(["2023-01-01", "2023-01-01", "2023-02-01", "2023-02-01", "2023-03-01"]).to_period("M"),
})

orders_with_cohort = orders.merge(customers, on="customer_id")
orders_with_cohort["order_month"] = orders_with_cohort["order_date"].dt.to_period("M")
orders_with_cohort["months_since_signup"] = (
    orders_with_cohort["order_month"] - orders_with_cohort["signup_month"]
).apply(lambda x: x.n)

cohort_pivot = orders_with_cohort.pivot_table(
    index="signup_month", columns="months_since_signup", values="customer_id", aggfunc=pd.Series.nunique
)
print("Cohort retention matrix (unique active customers):\n", cohort_pivot)

cohort_sizes = customers.groupby("signup_month")["customer_id"].nunique()
retention_pct = cohort_pivot.div(cohort_sizes, axis=0).round(3)
print("\nRetention % matrix:\n", retention_pct)


### Expected Output (verified on pandas 3.0.2)
```
Cohort retention matrix (unique active customers):
 months_since_signup   -1    0    1    2
signup_month
2023-01              NaN  2.0  1.0  2.0
2023-02              1.0  1.0  NaN  NaN
2023-03              NaN  1.0  NaN  NaN

Retention % matrix:
 months_since_signup   -1    0    1    2
signup_month
2023-01              NaN  1.0  0.5  1.0
2023-02              0.5  0.5  NaN  NaN
2023-03              NaN  1.0  NaN  NaN
```

### Reading a genuinely surprising real result — negative `months_since_signup`
Notice a `-1` column -- this means a customer's `order_month` was **before** their recorded
`signup_month`. In this synthetic data, that's an artifact of how the example was constructed,
but in a **real** dataset, `months_since_signup = -1` would be a genuine, important data-quality
red flag (Chapter 51's validity dimension): an order recorded before the customer's signup date
is either a data entry error, a signup-date backfill issue, or a sign that "signup_date" doesn't
mean what the analysis assumes it means. **A cohort analysis should always sanity-check for
negative "time since cohort start" values before trusting the retention percentages** -- this
notebook surfaced that check by actually running the numbers rather than assuming clean input.

The `2023-01` cohort's retention pattern (`100%` at month 0, `50%` at month 1, back to `100%`
at month 2) is a real, if small-sample, example of retention **not** monotonically declining --
customer 1 skipped month 1 entirely (ordered in January, then not again until March) but
returned by month 2, exactly the kind of "lost then won back" pattern a monthly retention curve
should be able to show, not force into a strictly decreasing shape.

### Interview Insight
> "In a cohort retention table, what would a negative 'months since signup' value indicate, and
> what would you check?"
An order (or other tracked activity) recorded **before** the customer's recorded signup date --
this is a data quality red flag, not a normal analytical result. Investigate whether the signup
date field is populated correctly (e.g., a backfilled or delayed signup timestamp), whether
there's a timezone/date-boundary mismatch between the two source systems, or whether "signup"
is being measured inconsistently across the pipeline.


---
# CHAPTER 71 — ML Feature Engineering Masterclass

## Concept
A consolidated, ready-reference feature catalog -- assembling frequency/count/recency/monetary
features (RFM's building blocks, generalized) with the exact leakage-risk discipline from
Chapter 39, applied at production density.


In [ ]:
txn = pd.DataFrame({
    "customer_id": [1, 1, 1, 2, 2, 3, 3, 3, 3],
    "date": pd.to_datetime(["2023-01-01", "2023-01-15", "2023-02-01", "2023-01-05", "2023-02-10",
                              "2023-01-01", "2023-01-10", "2023-01-20", "2023-02-01"]),
    "amount": [50, 60, 70, 100, 120, 20, 25, 30, 35],
})
reference_date = pd.Timestamp("2023-02-15")

features = txn.groupby("customer_id").agg(
    n_transactions=("amount", "count"),          # frequency feature
    total_spend=("amount", "sum"),                 # monetary feature
    avg_order_value=("amount", "mean"),              # monetary feature
    first_txn=("date", "min"),
    last_txn=("date", "max"),
)
features["days_since_last_txn"] = (reference_date - features["last_txn"]).dt.days   # recency feature
features["customer_age_days"] = (reference_date - features["first_txn"]).dt.days     # tenure feature
features["purchase_frequency"] = (
    features["n_transactions"] / (features["customer_age_days"] / 30).replace(0, np.nan)
)   # a derived RATIO feature: transactions per 30-day period

print(features)


### Expected Output (verified on pandas 3.0.2)
```
             n_transactions  total_spend  avg_order_value  first_txn   last_txn  days_since_last_txn  customer_age_days  purchase_frequency
customer_id
1                         3          180             60.0 2023-01-01 2023-02-01                   14                 31            2.903226
2                         2          220            110.0 2023-01-05 2023-02-10                    5                 36            1.666667
3                         4          110             27.5 2023-01-01 2023-02-01                   14                 31            3.870968
```

### The feature dictionary, with explicit leakage risk on each, per Chapter 39's discipline
| Feature | Business meaning | Leakage risk |
|---|---|---|
| `n_transactions` | Engagement volume | **Low** -- computed from historical data up to `reference_date`, never future data, since the input `txn` table is itself bounded to observed history |
| `total_spend` / `avg_order_value` | Customer value | **Low**, same reasoning -- but see the caveat below |
| `days_since_last_txn` | Recency/churn signal | **Low** if `reference_date` is fixed and genuinely in the past relative to any prediction use case |
| `purchase_frequency` | Normalized engagement rate | **Low** -- a ratio of two already-safe features |

**The caveat that matters**: all of these are safe **only if** `txn` itself is correctly
bounded to data available as of `reference_date` for each customer -- if this feature table were
built for a *training* dataset spanning many different historical prediction dates (not one
single `reference_date` for everyone), each customer's features would need to be computed
relative to **their own** specific historical cutoff, not one global date, or the exact Chapter
42 leakage pattern (group aggregation computed before/independent of the correct temporal
boundary) reappears.

### Interview Insight
> "You've engineered `total_spend` per customer from their full transaction history. Under what
> circumstance could this seemingly safe feature still leak information in a training dataset?"
If different training examples correspond to different historical prediction dates (e.g.,
predicting churn for various customers at various points in their lifecycle, not all "today"),
and `total_spend` is computed using **all** of a customer's transactions rather than only those
before that specific example's own prediction date -- this reintroduces the exact "using future
information relative to a specific row" leakage pattern from Chapter 42, just with the
leakage boundary being per-example instead of a single global train/test split date.


---
# CHAPTER 72 — Pandas for Experimentation

## Concept
Model experiment tracking as ordinary DataFrame analysis -- sorting, ranking, and grouping
experiment results is just `groupby`/`sort_values`/`rank` applied to a new kind of "business"
data: your own model runs.


In [ ]:
experiments = pd.DataFrame({
    "model": ["LogReg", "RandomForest", "XGBoost", "LogReg", "RandomForest"],
    "run_id": [1, 1, 1, 2, 2],
    "accuracy": [0.82, 0.88, 0.91, 0.83, 0.87],
    "precision": [0.80, 0.86, 0.90, 0.81, 0.85],
    "recall": [0.78, 0.85, 0.89, 0.79, 0.84],
    "train_time_sec": [2.1, 15.3, 22.7, 2.0, 14.9],
})
experiments["f1"] = 2 * (experiments["precision"] * experiments["recall"]) / (experiments["precision"] + experiments["recall"])

best_per_model = experiments.sort_values("accuracy", ascending=False).groupby("model").head(1)
print("Best run per model:\n", best_per_model.sort_values("accuracy", ascending=False))

model_ranking = experiments.groupby("model")["accuracy"].mean().rank(ascending=False)
print("\nModel ranking by avg accuracy:\n", model_ranking.sort_values())


### Expected Output (verified on pandas 3.0.2)
```
Best run per model:
           model  run_id  accuracy  precision  recall  train_time_sec        f1
2       XGBoost       1      0.91       0.90    0.89            22.7  0.894972
1  RandomForest       1      0.88       0.86    0.85            15.3  0.854971
3        LogReg       2      0.83       0.81    0.79             2.0  0.799875

Model ranking by avg accuracy:
 model
XGBoost         1.0
RandomForest    2.0
LogReg          3.0
Name: accuracy, dtype: float64
```

### The trade-off this table makes visible, that accuracy alone hides
XGBoost wins on accuracy (`0.91`) and F1 (`0.895`), but also takes **10.8x longer to train**
than LogReg (`22.7s` vs `2.0s`) for roughly `0.08` more accuracy -- whether that trade-off is
worth it depends entirely on the deployment context (a one-time offline training job vs. a
pipeline that retrains hourly). **This is exactly why an experiment-tracking table should
always include cost/resource columns alongside performance metrics** -- reporting only accuracy
rankings (as `model_ranking` alone does) would silently hide this real trade-off.

### Interview Insight
> "You're comparing three models and XGBoost has the highest accuracy. What else would you want
> in the comparison before recommending it for production?"
Training time and (if available) inference latency/resource cost -- a marginal accuracy gain
that costs substantially more compute/time to train or serve may not be worth it depending on
the production constraints (retraining frequency, latency SLAs, infrastructure cost). An
experiment-tracking table should surface these trade-offs directly, not just rank by the single
metric being optimized.


---
# CHAPTER 73 — Pandas for Model Error Analysis

## Concept
Given `actual`/`prediction`/`probability`, build a full error-analysis DataFrame -- classifying
every row's outcome and examining error rates by segment, connecting Pandas directly to ML
model evaluation.


In [ ]:
predictions = pd.DataFrame({
    "actual": [1, 0, 1, 1, 0, 0, 1, 0, 1, 0],
    "prediction": [1, 0, 0, 1, 1, 0, 1, 0, 0, 0],
    "probability": [0.91, 0.12, 0.45, 0.88, 0.62, 0.05, 0.95, 0.20, 0.48, 0.10],
    "segment": ["A", "A", "B", "B", "A", "B", "A", "B", "A", "B"],
})

predictions["outcome"] = np.select(
    [
        (predictions["actual"] == 1) & (predictions["prediction"] == 1),
        (predictions["actual"] == 0) & (predictions["prediction"] == 0),
        (predictions["actual"] == 0) & (predictions["prediction"] == 1),
        (predictions["actual"] == 1) & (predictions["prediction"] == 0),
    ],
    ["TP", "TN", "FP", "FN"],
    default="unknown",
)
print(predictions)
print("\nOutcome counts:\n", predictions["outcome"].value_counts())


### Expected Output (verified on pandas 3.0.2)
```
   actual  prediction  probability segment outcome
0       1           1         0.91       A      TP
1       0           0         0.12       A      TN
2       1           0         0.45       B      FN
3       1           1         0.88       B      TP
4       0           1         0.62       A      FP
5       0           0         0.05       B      TN
6       1           1         0.95       A      TP
7       0           0         0.20       B      TN
8       1           0         0.48       A      FN
9       0           0         0.10       B      TN

Outcome counts:
outcome
TN    4
TP    3
FN    2
FP    1
Name: count, dtype: int64
```

## Error rate by segment — where does the model actually struggle?


In [ ]:
segment_outcomes = predictions.groupby("segment")["outcome"].value_counts(normalize=True)
print("Outcome rate by segment:\n", segment_outcomes)

false_negatives = predictions[predictions["outcome"] == "FN"]
print("\nFalse negatives (often the most costly error type):\n", false_negatives)


### Expected Output (verified on pandas 3.0.2)
```
Outcome rate by segment:
segment  outcome
A        TP         0.4
         TN         0.2
         FP         0.2
         FN         0.2
B        TN         0.6
         FN         0.2
         TP         0.2
Name: proportion, dtype: float64

False negatives (often the most costly error type):
   actual  prediction  probability segment outcome
2       1           0         0.45       B      FN
8       1           0         0.48       A      FN
```

### Reading these results as the actual investigative payoff
Segment A shows a genuine `FP` rate (`0.2`, one false positive) that segment B doesn't have at
all (`0.0`) -- a segment-specific miscalibration worth investigating (is the model systematically
over-predicting positives for segment A?). Both false negatives (`0.45` and `0.48`
probability) sit **just below** the implicit `0.5` decision threshold -- this is a real,
actionable pattern: these aren't confidently-wrong predictions, they're borderline cases the
model was genuinely uncertain about, which is a very different remediation story (consider
threshold tuning, or these examples may benefit from additional features/data) than a
confidently-wrong false negative (e.g., predicted `0.05` when actually `1`) would suggest.

### Interview Insight
> "Two false negatives both have prediction probabilities just below your decision threshold.
> What does this suggest, and how might you respond?"
The model wasn't confidently wrong -- it was genuinely uncertain, and the fixed threshold
happened to fall on the wrong side of that uncertainty for these specific cases. This is a
different situation than a confident, large-margin misclassification (which suggests a genuine
gap in what the model has learned) -- near-threshold errors are the natural candidates for
threshold tuning (if the cost of false negatives is high, lowering the threshold captures more
of these borderline true positives, at the cost of more false positives elsewhere) rather than
necessarily indicating a need for new features or more training data.


---
# CHAPTER 74 — Pandas for Model Monitoring

## Concept
Simulated production prediction data, analyzed for **drift** -- feature distribution shift
between a baseline (training-time) reference and current production data, using nothing but
`.describe()` and `.value_counts()` comparisons.


In [ ]:
rng = np.random.default_rng(0)

baseline = pd.DataFrame({
    "feature_a": rng.normal(50, 10, 1000),
    "feature_b": rng.choice(["X", "Y", "Z"], 1000, p=[0.5, 0.3, 0.2]),
})
production = pd.DataFrame({
    "feature_a": rng.normal(58, 12, 1000),
    "feature_b": rng.choice(["X", "Y", "Z"], 1000, p=[0.3, 0.3, 0.4]),
})

print("Baseline feature_a stats:\n", baseline["feature_a"].describe())
print("\nProduction feature_a stats:\n", production["feature_a"].describe())
print("\nMean shift:", production["feature_a"].mean() - baseline["feature_a"].mean())


### Expected Output (verified on pandas 3.0.2)
```
Baseline feature_a stats:
 count    1000.000000
mean       49.519717
std         9.772417
min        11.005783
25%        43.095664
50%        49.251781
75%        56.151463
max        80.660367
Name: feature_a, dtype: float64

Production feature_a stats:
 count    1000.000000
mean       57.484906
std        11.766669
min        21.862599
25%        49.788211
50%        56.612443
75%        65.527261
max        93.670892
Name: feature_a, dtype: float64

Mean shift: 7.965188651683853
```

### Reading a ~0.8-standard-deviation mean shift as a real monitoring signal
Production's `feature_a` mean (`57.48`) sits nearly **8 units above** baseline's (`49.52`) --
relative to baseline's own standard deviation (`9.77`), that's roughly a `0.8` standard
deviation shift, a magnitude most production monitoring systems would flag as worth
investigating (not necessarily alarming on its own, but a real, non-trivial distributional
change from what the model was trained on). Production's spread also widened (`std` `11.77` vs
`9.77`) -- the feature isn't just shifted, it's become genuinely more variable too, which can
independently degrade model calibration even if the mean shift alone were judged acceptable.

## Categorical feature drift


In [ ]:
baseline_dist = baseline["feature_b"].value_counts(normalize=True).sort_index()
prod_dist = production["feature_b"].value_counts(normalize=True).sort_index()

drift_table = pd.DataFrame({"baseline": baseline_dist, "production": prod_dist})
drift_table["abs_diff"] = (drift_table["production"] - drift_table["baseline"]).abs()
print(drift_table)


### Expected Output (verified on pandas 3.0.2)
```
            baseline  production  abs_diff
feature_b
X             0.533       0.300     0.233
Y             0.264       0.312     0.048
Z             0.203       0.388     0.185
```

### Reading the categorical drift precisely
Category `X`'s share dropped by `23.3` percentage points (`53.3%` -> `30.0%`), while `Z` gained
`18.5` points (`20.3%` -> `38.8%`) -- `Y` stayed comparatively stable (`4.8` point shift). This
kind of per-category breakdown (not just an aggregate "the distribution changed" flag) is what
lets an engineer distinguish "the whole distribution drifted uniformly" from "one or two
specific categories are driving the shift" -- the latter usually points toward a more specific,
investigable root cause (a new user segment appearing, a source system change affecting one
category disproportionately) than the former.

### Interview Insight
> "How would you detect feature drift in a production ML system using only Pandas, without a
> dedicated monitoring library?"
Compare summary statistics (`.describe()`) between a stored baseline (training-time) reference
sample and a current production sample for numeric features -- a substantial shift in mean
relative to the baseline's own standard deviation is a practical, interpretable signal. For
categorical features, compare `.value_counts(normalize=True)` distributions directly, computing
the per-category absolute difference -- this surfaces both *that* drift occurred and *which*
specific categories are driving it, which a single aggregate drift score would hide.


---
# CHAPTER 75 — Capstone Project

**This is a real assignment. Attempt it yourself before checking your solution against the
patterns from Chapters 1-74 — do not just read this and move on.**

## Business Problem
You work at a subscription analytics company. Leadership wants a complete customer health and
churn-risk report, built entirely in Pandas, that can run as a scheduled nightly job.

## Dataset Description
Three raw tables, each with realistic data quality problems you must discover and handle:
- **`customers`**: `customer_id`, `signup_date` (multiple formats), `region` (inconsistent
  casing/whitespace), `plan_type`
- **`transactions`**: `transaction_id`, `customer_id`, `transaction_date`, `amount` (some
  stored as text, some negative -- refunds), `product`
- **`support_tickets`**: `ticket_id`, `customer_id`, `created_date`, `resolved_date` (some
  missing -- unresolved tickets), `priority`

## Data Quality Problems to Discover and Handle
Duplicate customer records (investigate before dropping -- Chapter 13/37's discipline);
inconsistent region naming; mixed date formats; a `customer_id` in `transactions` that doesn't
exist in `customers` (an orphaned foreign key -- decide how to handle, and justify the decision);
negative transaction amounts representing refunds (decide whether to net them against gross
revenue or track separately, and justify).

## Business Questions to Answer
Who are the top 10% of customers by lifetime value? What's the relationship between support
ticket volume/priority and churn risk? Which regions and plan types show the highest
churn-risk concentration? Build an RFM segmentation and cross-reference it against support
ticket history.

## ML Objective
Prepare a model-ready churn dataset: define churn (you must choose and justify a definition,
e.g., "no transaction in the last 90 days"), engineer at minimum: RFM features, support ticket
features (volume, average resolution time, unresolved ticket count), tenure, and region/plan
type encodings -- with an explicit leakage-risk label on every engineered feature, per Chapter
39's discipline.

## Constraints
The full pipeline must run correctly even if a table has zero rows for some customers (a
customer with no transactions, or no support tickets -- must not crash or silently drop that
customer from the final output). Every date-based feature must be computed relative to a fixed,
explicit `reference_date` parameter, never `pd.Timestamp.now()` (Chapter 56's discipline).

## Expected Deliverables
A single model-ready DataFrame (one row per customer), a written data quality investigation
log (what you found, what you decided, why), and a validated leakage-risk table for every
engineered feature.

## Evaluation Criteria
Correctness of joins (verified row counts, not assumed) · defensible data quality decisions ·
leakage-safety of every feature · code organized as named, reusable functions (Chapter 54) ·
test coverage for at least the core cleaning/feature functions (Chapter 61) · clear
justification of every non-obvious decision.

---

*A full reference solution is intentionally not included -- building it yourself, hitting the
orphaned-foreign-key decision point and the churn-definition judgment call, and resolving them
defensibly is the actual point of a capstone. If you complete an attempt and want it reviewed
the way a senior data scientist would review a pull request, share it in your next message.*


---
# CHAPTER 76 — Mock Pandas Interview

A mock interview only works as a real back-and-forth -- this chapter can't simulate that inside
a static notebook, so consider this the opening of the interview rather than its content.

**If you'd like to run the mock interview, say so in your next message** (e.g., "start the mock
interview" or "quiz me at the advanced level"), and it will proceed exactly as this chapter's
original brief specified:

- Questions asked one at a time, mixing Pandas theory, coding, debugging, data cleaning, SQL
  comparison, EDA, feature engineering, performance, ML scenarios, and business analytics --
  drawn from the full 76-chapter arc you've just completed.
- Your answer is scored, mistakes are identified, the correct answer is explained, and a harder
  follow-up is asked -- continuing until you demonstrate mastery, not stopping after one
  question.
- You can request a specific difficulty band or a specific topic area (e.g., "only groupby and
  merge questions" or "only data leakage scenarios") if you want to target weak spots directly.

## Final mastery self-assessment — answer honestly before requesting the mock interview
```
Level 1 - Understanding: Can you explain alignment, the agg/transform/filter distinction, and
                          the merge row-multiplication rule to someone else, from memory?
Level 2 - Coding:        Can you write correct groupby/merge/pivot code for a described task
                          without checking documentation for basic operations?
Level 3 - Debugging:     Given unexpected row counts after a merge, or a silently-wrong
                          groupby result, do you know within seconds which Chapter 25/47
                          category it falls into?
Level 4 - Data Quality:  Can you independently investigate a messy dataset -- duplicates,
                          inconsistent categories, mixed date formats -- and defend your
                          cleaning decisions, not just execute a checklist?
Level 5 - Leakage:       Can you identify a leaky feature (temporal, group-aggregation-based,
                          or full-dataset-statistic-based) on sight, in code you didn't write?
Level 6 - Business:      Have you connected every technique to a real analytical or ML use case
                          (Chapters 55-59, 70-74) rather than just abstract syntax?
Level 7 - Expert:        Can you reason about an UNFAMILIAR data problem you've never seen
                          before using the mental models this course built -- including knowing
                          when to recommend moving beyond Pandas entirely?
```


---
## COURSE COMPLETE — Chapters 1 through 76

```
PANDAS MASTERY PROGRESS

[x] Mental Model                    [x] MultiIndex                      [x] Design Patterns
[x] Series                          [x] Merge                            [x] Method Chaining
[x] DataFrame                       [x] Join Debugging                    [x] Pipeline Cleaning
[x] Indexing (.loc/.iloc/.at/.iat)  [x] Concat                             [x] Customer Analytics Project
[x] Filtering                       [x] Reshaping (melt/pivot)              [x] Churn Data Prep Project
[x] Data Types                      [x] Pivot Tables                         [x] Sales Analytics Project
[x] Reading Data                    [x] Time Series (.dt)                     [x] Fraud Data Prep Project
[x] Writing Data                    [x] Time Series Resampling                 [x] Employee Analytics Project
[x] Data Inspection                 [x] shift/diff/pct_change/ewm               [x] Project Structure
[x] Missing Values                  [x] Window Operations (rolling)              [x] Pandas Testing
[x] Duplicates                      [x] Ranking                                   [x] Interview Mastery
[x] Sorting                         [x] Statistical Operations                     [x] Coding Interview
[x] Modifying Data                  [x] Value Counts                                 [x] Real-World Scenarios
[x] String Operations               [x] Unique Values                                 [x] Pandas vs SQL/Polars/Spark
[x] Categorical Data                [x] Data Cleaning Project                          [x] Performance Engineering
[x] Apply/Map/Vectorization         [x] EDA Workflow                                    [x] Advanced Patterns
[x] Conditional Columns             [x] Feature Engineering                              [x] RFM/Cohort/Retention
[x] GroupBy                         [x] Pandas for ML (X/y)                               [x] ML Feature Masterclass
[x] GroupBy Advanced                [x] Pandas + Scikit-learn                              [x] Experimentation
                                     [x] Data Leakage                                       [x] Error Analysis
                                     [x] SettingWithCopyWarning / CoW                       [x] Model Monitoring
                                     [x] Index Management                                   [x] Capstone
                                     [x] Data Validation                                    [x] Mock Interview (ready)
                                     [x] Data Quality Framework
```

You've gone from "I know NumPy" to a complete, verified command of Pandas: labeled-data
mechanics, the groupby/merge/reshape core that powers nearly all real analytical work, a
current, accurate understanding of pandas 3.0's genuine structural improvements (mandatory
Copy-on-Write, the new `str` dtype, read-only `.to_numpy()`, clearer merge/dtype errors -- not
outdated pre-3.0 folklore), data leakage discipline proven with real numbers rather than just
asserted, and eight full applied projects spanning customer analytics, churn, sales, fraud, HR,
RFM/cohort analysis, experimentation, and monitoring.

**Where to go from here, in order of value:**
1. Attempt Chapter 75's capstone for real -- this is where remaining gaps surface, especially
   the judgment calls (orphaned foreign keys, churn definition) that don't have one "correct"
   answer.
2. Say "start the mock interview" and go through it honestly, without peeking at earlier
   chapters' answers.
3. Take one of this course's own applied projects (Chapter 55-59, 70-74) and rebuild it once
   more, closed-book, on a dataset you generate yourself rather than reusing the one provided --
   retention testing beats passive re-reading every time.
